# Notebook 04 — Methods and Reproducibility

**Module:** 18 — Scientific Writing and Open Science  
**Tier:** 2 — Working competence  
**Notebook:** 4 of 12  
**Time estimate:** 75 minutes

> A methods section has exactly one goal: enable an independent lab to
> replicate your result from scratch, without contacting you.
> If they would need to email you, the section has failed.

---
## Step 1 — Motivation

The reproducibility crisis in biology (Begley & Ellis 2012, Baker 2016) has its
roots largely in methods sections that omit critical parameters, software versions,
or data provenance. In computational biology specifically, >70% of papers in a
2019 survey could not be reproduced from their stated methods alone (Trisovic et al.
2022). The methods section is also where reviewers check whether your claims are
technically justified — a methods section with "we used standard tools" is an
immediate red flag to a specialist reviewer.

---
## Step 2 — Methods Section Structure

For a computational biology paper, the methods section typically has 5–7 subsections:

| Subsection | What to include |
|-----------|----------------|
| **Datasets** | Source, version, accession numbers, n, split strategy, preprocessing steps |
| **Model / algorithm** | Full description, architecture, parameters, loss function |
| **Training / fitting** | Optimizer, learning rate, batch size, n epochs, early stopping rule, hardware |
| **Evaluation** | Metrics, validation strategy (k-fold, held-out, cross-dataset), n replicates |
| **Statistical analysis** | Tests used, multiple testing correction, confidence intervals, p-value thresholds |
| **Baselines** | Every comparison method, its parameters, its version, where its code was obtained |
| **Software and availability** | All software with versions; GitHub URL; DOI (via Zenodo); data accession |

---
## Step 3 — The FAIR Principles

FAIR = **Findable, Accessible, Interoperable, Reusable** (Wilkinson et al. 2016, *Nature Scientific Data*).

| Principle | What it means for your methods section |
|-----------|---------------------------------------|
| **Findable** | Every dataset and software tool has a DOI or accession number in the text |
| **Accessible** | All datasets are publicly available (or the embargo reason is stated) |
| **Interoperable** | File formats are standard (FASTQ, BAM, CSV, HDF5) — not proprietary |
| **Reusable** | Code is licensed (MIT/Apache/GPL), documented, and archived at a persistent URL |

**Minimum FAIR for a computational paper:**
- GitHub repository with a README that includes a one-command reproduce instruction
- A Zenodo DOI for the repository snapshot (so the GitHub URL isn't a broken link in 10 years)
- All datasets cited with GEO/EBI/ENCODE accession numbers, not just "publicly available data"

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Optional

# ---- Methods completeness auditing ----

@dataclass
class MethodsAudit:
    """Checklist for computational biology methods section completeness."""

    # Dataset items
    dataset_source: bool = False          # Named source (GEO, ENCODE, etc.)
    dataset_version: bool = False          # Version / accession number
    dataset_n: bool = False                # Sample size, train/test split
    dataset_preprocessing: bool = False    # All preprocessing steps listed

    # Model items
    model_architecture: bool = False       # Full architecture described
    model_hyperparameters: bool = False    # All tuned hyperparameters stated
    model_loss: bool = False               # Loss function defined mathematically
    model_optimizer: bool = False          # Optimizer + learning rate

    # Evaluation items
    eval_metric: bool = False              # All metrics defined
    eval_validation: bool = False          # Validation strategy (CV, held-out, etc.)
    eval_nreplicates: bool = False         # Number of independent runs
    eval_stats: bool = False               # Statistical test for comparisons

    # Reproducibility items
    repro_software_versions: bool = False  # All software with version numbers
    repro_random_seed: bool = False        # Random seed(s) stated
    repro_hardware: bool = False           # GPU/CPU, memory (if timing matters)
    repro_code_url: bool = False           # GitHub URL or DOI
    repro_data_url: bool = False           # Data accession or DOI

    def score(self):
        fields = [getattr(self, f.name) for f in self.__dataclass_fields__.values()]
        return sum(fields), len(fields)

    def missing(self):
        return [name for name, f in self.__dataclass_fields__.items()
                if not getattr(self, name)]

def audit_methods_text(text):
    """Heuristically score a methods section text against the checklist."""
    audit = MethodsAudit()
    t = text.lower()

    # Dataset checks
    audit.dataset_source = any(x in t for x in ['geo', 'encode', 'ebi', 'genbank',
                                                    'accession', 'downloaded from'])
    audit.dataset_version = bool(re.search(r'(gse|gsm|srr|prjna|encsr)\d{4,}', t, re.IGNORECASE))
    audit.dataset_n = bool(re.search(r'n\s*=\s*\d+|\d+\s*(samples|sequences|sequences|patients)', t))
    audit.dataset_preprocessing = any(x in t for x in ['trimmed', 'filtered', 'normalized',
                                                           'preprocessed', 'quality control'])
    # Model checks
    audit.model_architecture = any(x in t for x in ['layer', 'conv', 'lstm', 'transformer',
                                                        'hidden units', 'neurons', 'parameters'])
    audit.model_hyperparameters = bool(re.search(r'learning rate|lr\s*=|dropout|kernel size', t))
    audit.model_loss = any(x in t for x in ['cross-entropy', 'bce', 'mse', 'loss function',
                                               'binary cross', 'log-likelihood'])
    audit.model_optimizer = any(x in t for x in ['adam', 'sgd', 'adagrad', 'rmsprop'])

    # Evaluation checks
    audit.eval_metric = any(x in t for x in ['auroc', 'auc', 'accuracy', 'f1', 'precision',
                                                'recall', 'pearson', 'spearman'])
    audit.eval_validation = any(x in t for x in ['cross-validation', 'held-out', 'test set',
                                                    'k-fold', 'leave-one-out', 'train/test'])
    audit.eval_nreplicates = bool(re.search(r'\d+\s*independent|\d+\s*replicat|\d+\s*runs', t))
    audit.eval_stats = any(x in t for x in ['t-test', 'wilcoxon', 'mann-whitney', 'anova',
                                               'bonferroni', 'fdr', 'p-value', 'confidence interval'])

    # Reproducibility checks
    audit.repro_software_versions = bool(re.search(
        r'python\s+\d|pytorch\s+\d|tensorflow\s+\d|version\s+\d|v\d+\.\d+', t))
    audit.repro_random_seed = any(x in t for x in ['random seed', 'seed =', 'reproducibility',
                                                      'set.seed'])
    audit.repro_hardware = any(x in t for x in ['gpu', 'cpu', 'nvidia', 'v100', 'a100', 'tpu'])
    audit.repro_code_url = any(x in t for x in ['github', 'zenodo', 'doi', 'https://', 'available at'])
    audit.repro_data_url = any(x in t for x in ['accession', 'repository', 'zenodo',
                                                    'available at', 'supplementary'])
    return audit

# Synthetic methods sections
METHODS_INCOMPLETE = """
We trained a convolutional neural network on ChIP-seq data from ENCODE.
Sequences were preprocessed and used to train the model with the Adam optimizer.
We evaluated performance using AUROC. Results are available on GitHub.
"""

METHODS_COMPLETE = """
Datasets. ChIP-seq data for 690 human transcription factors was downloaded from
ENCODE (accession ENCSR000AKF; release 2022-01-01). Positive sequences were 200 bp
centered on ChIP-seq peaks; negatives were GC-matched dinucleotide-shuffled controls.
All sequences were split into train (80%), validation (10%), and test (10%) by
chromosome to prevent data leakage (n = 12,400 train, 1,550 validation, 1,550 test
per TF).

Model. DeepBind-v2 is a convolutional neural network with architecture:
Conv1d(in=4, out=32, kernel=7) → BatchNorm1d(32) → ReLU → AdaptiveMaxPool1d(1)
→ Dropout(0.3) → Linear(32, 1) → Sigmoid. Binary cross-entropy loss. Total
trainable parameters: 6,977. Training used Adam optimizer (lr=1e-3, betas=(0.9, 0.999))
for a maximum of 50 epochs with early stopping (patience=5, monitor=validation AUROC).
All experiments were run on a single NVIDIA A100 GPU.

Evaluation. Classification performance was measured by AUROC and AUPRC on the held-out
test set. For each TF, 5 independent training runs with different random seeds (42,
43, 44, 45, 46) were performed; we report mean ± standard deviation. Comparison to
the PWM baseline (FIMO v5.4.1, JASPAR 2022 motifs) used a two-sided Wilcoxon signed-rank
test with Bonferroni correction for 690 TFs; significance threshold α = 0.05.

Software and availability. All code is written in Python 3.11, using PyTorch 2.1.0,
numpy 1.26.0, and scikit-learn 1.3.0. Random seeds set via numpy.random.seed and
torch.manual_seed. Full code is available at https://github.com/example/deepbind-v2
(DOI: 10.5281/zenodo.XXXXXXX). Raw ChIP-seq data are available from ENCODE
(accession above); processed sequences are deposited at https://zenodo.org/XXXXXXX.
"""

for name, methods in [('INCOMPLETE', METHODS_INCOMPLETE), ('COMPLETE', METHODS_COMPLETE)]:
    audit = audit_methods_text(methods)
    score, total = audit.score()
    print(f'\n=== {name} methods ({score}/{total} = {score/total:.0%}) ===')
    missing = audit.missing()
    if missing:
        print(f'  Missing: {missing[:5]}{"..." if len(missing)>5 else ""}')
    else:
        print('  All criteria met.')

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# Panel A: Methods completeness scores by category
ax = axes[0]
categories = ['Dataset\n(4 items)', 'Model\n(4 items)', 'Evaluation\n(4 items)', 'Reproducibility\n(5 items)']

incomplete_audit = audit_methods_text(METHODS_INCOMPLETE)
complete_audit   = audit_methods_text(METHODS_COMPLETE)

def category_scores(audit):
    return [
        sum([audit.dataset_source, audit.dataset_version, audit.dataset_n, audit.dataset_preprocessing]) / 4,
        sum([audit.model_architecture, audit.model_hyperparameters, audit.model_loss, audit.model_optimizer]) / 4,
        sum([audit.eval_metric, audit.eval_validation, audit.eval_nreplicates, audit.eval_stats]) / 4,
        sum([audit.repro_software_versions, audit.repro_random_seed, audit.repro_hardware,
               audit.repro_code_url, audit.repro_data_url]) / 5,
    ]

inc_scores  = category_scores(incomplete_audit)
comp_scores = category_scores(complete_audit)

x = np.arange(len(categories))
w = 0.35
ax.bar(x - w/2, inc_scores,  w, label='Incomplete', color='tomato',    edgecolor='black', alpha=0.8)
ax.bar(x + w/2, comp_scores, w, label='Complete',   color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 1.15); ax.set_ylabel('Category completeness (0–1)')
ax.axhline(1.0, color='grey', ls=':', lw=0.8)
ax.set_title('A. Methods completeness by category')
ax.legend(fontsize=9)

# Panel B: FAIR principles visual
ax = axes[1]
ax.axis('off')
fair_data = [
    ('F — Findable', '#4e79a7',
     '• DOI for every dataset\n• Software has a persistent URL\n• Zenodo archive for code'),
    ('A — Accessible', '#f28e2b',
     '• Datasets publicly available\n• Code runs from a clean clone\n• README with install steps'),
    ('I — Interoperable', '#59a14f',
     '• Standard formats (FASTQ, CSV, HDF5)\n• No proprietary tools required\n• Documented file schemas'),
    ('R — Reusable', '#e15759',
     '• MIT/Apache license on code\n• Data license clearly stated\n• Documented parameters'),
]
for i, (title, color, text) in enumerate(fair_data):
    y = 0.95 - i * 0.23
    ax.text(0.05, y, title, transform=ax.transAxes, fontsize=10, fontweight='bold', color=color)
    ax.text(0.05, y - 0.08, text, transform=ax.transAxes, fontsize=8.5,
              bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.15))
ax.set_title('B. FAIR principles for computational biology\n(Wilkinson et al. 2016)')

# Panel C: Methods section word count targets
ax = axes[2]
subsections = ['Datasets', 'Model /\nAlgorithm', 'Training /\nFitting', 'Evaluation', 'Statistics', 'Software\n& Availability']
target_min = [80,  100, 60,  80,  60,  50]
target_max = [150, 200, 120, 150, 100, 100]
ax.barh(range(len(subsections)), target_max, color='steelblue', alpha=0.4, label='Max')
ax.barh(range(len(subsections)), target_min, color='steelblue', alpha=0.8, label='Min')
ax.set_yticks(range(len(subsections))); ax.set_yticklabels(subsections, fontsize=9)
ax.set_xlabel('Word count')
ax.set_title('C. Word count targets per\nmethods subsection')
ax.legend(fontsize=9)

plt.suptitle('Module 18 NB04: Methods and Reproducibility', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('methods_reproducibility.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

See E04 in `exercises/README.md`: complete the methods audit checklist
for a real bioinformatics paper, then identify the two most critical
missing items.

---
## Step 10 — Quiz

1. List the 5–7 standard subsections of a computational biology methods section.
   What does each contain?
2. What are the FAIR principles? Give one concrete action for each letter
   that you could take for a GitHub repository.
3. A reviewer says: "The authors do not specify which version of HISAT2 was used,
   or whether the default splice-site database was employed." Is this a reasonable
   review comment? Why?
4. What is the minimum information required for a random experiment to be
   reproducible by another lab? (Think: data, code, environment, seed.)

---
## Step 12 — Reflection

> *[Run `audit_methods_text()` on the methods section of any paper you have
> read in the course so far. Report the score and the top 3 missing items.
> How would you rewrite one of those missing items?]*

---
*Next: `05_results_writing.ipynb`*